## Build your Own

### Test function Definition

In [1]:
def rosenbrock(params):
    x, y = params
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

def rosenbrock_analytic_gradient(params):
    x, y = params
    df_dx = -2 * (1 - x) - 400 * x * (y - x ** 2)
    df_dy = 200 * (y - x ** 2)
    return [df_dx, df_dy]

## Gradient descent

In [2]:
class GradientDescent:
    def __init__(self, lr = 0.001):
        self.lr = lr

    def step(self, params, grads):
        return [p - self.lr * g for p, g in zip(params, grads)]

## Gradient descent with motion

In [3]:
class SGDMomentum:
    def __init__(self, lr = 0.0001, momentum = 0.9):
        self.lr = lr
        self.momentum = momentum
        self.velocity = None

    def step(self, params, grads):
        if self.velocity is None:
            self.velocity = [0.0] * len(params)

        self.velocity = [
            self.momentum * v + g
            for v, g in zip(self.velocity, grads)
        ]

        return [p - self.lr * v for p, v in zip(params, self.velocity)]

## ADAM

Adam = **动量（一阶矩）** + **自适应学习率（二阶矩）** + **偏差修正**。

- 一阶矩 $m$：梯度的指数滑动平均 → 像 Momentum，方向更稳
- 二阶矩 $v$：梯度平方的指数滑动平均 → 像 RMSProp，陡的维度步子自动变小
- 偏差修正：$m,v$ 从 0 起步会偏小，除以 $1-\beta^t$ 拉正早期估计


In [5]:
class Adam:
    """
    设计原理：
      普通 GD:     p ← p - lr * g              （每维同一学习率，噪声大）
      Momentum:    用速度平滑 g                （稳方向）
      RMSProp:     用 g² 缩小「一直很大」的维 （自适应步长）
      Adam:        两者结合，并对早期偏差做修正

    每步计算：
      m_t = β1·m_{t-1} + (1-β1)·g_t          # 一阶矩：梯度均值
      v_t = β2·v_{t-1} + (1-β2)·g_t²         # 二阶矩：梯度方差（未中心化）
      m̂_t = m_t / (1-β1^t)                   # 偏差修正（t 小时分母 < 1，放大 m）
      v̂_t = v_t / (1-β2^t)
      p   ← p - lr · m̂_t / (√v̂_t + ε)        # 有效步长 ≈ lr / √v̂
    """

    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1      # 一阶矩衰减：越大越「记仇」久（常用 0.9）
        self.beta2 = beta2      # 二阶矩衰减：通常更接近 1（常用 0.999）
        self.epsilon = epsilon  # 防止除以 0
        self.m = None           # 一阶矩（动量方向）
        self.v = None           # 二阶矩（各维梯度尺度）
        self.t = 0              # 更新步数，用于偏差修正

    def step(self, params, grads):
        if self.m is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)

        self.t += 1

        # 1) 更新一阶矩：平滑梯度，减少单步噪声
        self.m = [
            self.beta1 * m + (1 - self.beta1) * g
            for m, g in zip(self.m, grads)
        ]

        # 2) 更新二阶矩：记录各维「梯度有多大」，用于自适应缩放
        self.v = [
            self.beta2 * v + (1 - self.beta2) * g ** 2
            for v, g in zip(self.v, grads)
        ]

        # 3) 偏差修正：m、v 初始化为 0，早期会被 β 拉向 0；除以 (1-β^t) 纠正
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]

        # 4) 参数更新：分子给方向，分母按该维历史梯度大小缩步长
        return [
            p - self.lr * mh / (vh ** 0.5 + self.epsilon)
            for p, mh, vh in zip(params, m_hat, v_hat)
        ]


## Adam vs AdamW

两者优化步骤几乎一样；差别在 **权重衰减（weight decay）怎么加**。

### 背景：想给参数「瘦身」

训练时常加正则，避免 $w$ 过大。常见写法是损失里加 $\frac{\lambda}{2}\|w\|^2$，梯度变成：

$$
g \leftarrow g + \lambda w
$$

这叫 **L2 正则**。对普通 SGD，L2 正则 ≈ 每步再乘一个衰减：$w \leftarrow (1 - lr\cdot\lambda)\,w - lr\cdot g$。

### Adam 里直接做 L2 会出问题

若把 $\lambda w$ 加进梯度再送进 Adam：

- $\lambda w$ 会进入一阶矩 $m$、二阶矩 $v$
- 自适应分母 $\sqrt{v}$ 会 **扭曲** 衰减强度：梯度大的维衰减变弱，小的维衰减变强
- 于是「名义上的 weight_decay」和真正施加在 $w$ 上的衰减不再一致

很多框架里的 `Adam(..., weight_decay=...)` 历史上就是这种 **耦合进梯度** 的做法。

### AdamW：把衰减从自适应更新里拆开

AdamW（Decoupled Weight Decay）做法是：

1. 仍用 **原始梯度** $g$ 更新 $m, v$（和 Adam 相同）
2. 参数更新时 **额外** 做一次与自适应无关的衰减：

$$
w \leftarrow w - lr \cdot \frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon} - lr \cdot \lambda \cdot w
$$

衰减项 $lr\cdot\lambda\cdot w$ 不经过 $m/v$，每维按同一 $\lambda$ 收缩 → 正则语义清晰，大模型里更常用。

| | Adam + L2（耦合） | AdamW（解耦） |
|--|--|--|
| $\lambda w$ 进 $m,v$？ | 是 | 否 |
| 衰减是否被 $\sqrt{v}$ 缩放 | 是 | 否 |
| 典型场景 | 旧代码 / 小任务 | Transformer 等现代训练默认 |

一句话：**AdamW ≈ Adam 的更新规则 + 单独、均匀的权重衰减；不是换了一套动量公式。**


In [ ]:
class AdamW(Adam):
    """AdamW = Adam 更新 + 解耦的权重衰减（不把 λw 加进梯度）。"""

    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=0.01):
        super().__init__(lr, beta1, beta2, epsilon)
        self.weight_decay = weight_decay

    def step(self, params, grads):
        # 注意：grads 保持原样，不要 grads + λ*w（那是耦合 L2）
        updated = super().step(params, grads)
        # 解耦衰减：与 √v 无关，每维同等收缩
        return [
            u - self.lr * self.weight_decay * p
            for u, p in zip(updated, params)
        ]


## Test And Comparison

In [8]:
def optimize(optimizer, func, grad_func, start, steps = 5000):
    params = list(start)
    history = [params[:]]

    for _ in range(steps):
        grads = grad_func(params)
        params = optimizer.step(params, grads)
        history.append(params[:])

    return history

start = [-1.0, 1.0]

gd_history = optimize(GradientDescent(lr=0.0005), rosenbrock, rosenbrock_analytic_gradient, start)

sgd_history = optimize(SGDMomentum(lr=0.0001, momentum=0.9), rosenbrock, rosenbrock_analytic_gradient, start)

adam_history = optimize(Adam(lr=0.01), rosenbrock, rosenbrock_analytic_gradient, start)

for name, history in [("GD", gd_history), ("SGD", sgd_history), ("Adam", adam_history)]:
    final = history[-1]
    loss = rosenbrock(final)
    print(f"{name:6s} --> x={final[0]:.6f}, y={final[1]:.6f}, loss={loss:.6f}")



GD     --> x=0.798131, y=0.636104, loss=0.040834
SGD    --> x=0.940412, y=0.884127, loss=0.003557
Adam   --> x=0.999997, y=0.999994, loss=0.000000


## Predefined in Pytorch

In [ ]:
import torch

model = torch.nn.Linear(784, 10)

sgd = torch.optim.SGD(model.parameters(), lr = 0.01, momentum= 0.9)

adam = torch.optim.Adam(model.parameters(), lr = 0.001)

adamw = torch.optim.AdamW(model.parameters(), lr = 0.001, weight_decay=0.01)